# Predictions Test Analysis

Analysis of model predictions on the NIST23 test set (9-fold cross-validation), joining:
- `data/predicted/nps_inchikey_nist23_nist23_cv{0..8}.zip` — model outputs and cosine similarity scores (one zip per CV fold)
- `data/proc/nist23/spec_df.pkl` — spectrum metadata
- `data/proc/nist23/mol_df.pkl` — molecule metadata
- `data/classyfire/nist23_mol_classyfire.pkl` — ClassyFire chemical taxonomy labels

All 9 CV folds are concatenated into a single DataFrame before analysis.

In [1]:
import gc
import pickle
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

DATA_ROOT = Path("../data")

## 1. Load Data

In [2]:
# Load and combine predictions from all 9 CV folds
cv_dfs = []
for cv_idx in range(9):
    zip_path = DATA_ROOT / f"predicted/nps_inchikey_nist23_nist23_cv{cv_idx}.zip"
    with zipfile.ZipFile(zip_path) as zf:
        pkl_name = next(n for n in zf.namelist() if n.endswith("predictions_test.pkl"))
        with zf.open(pkl_name) as f:
            cv_df = pickle.load(f)
    cv_df["cv_fold"] = cv_idx
    cv_dfs.append(cv_df)
    print(f"  cv{cv_idx}: {len(cv_df):,} rows")

pred_df = pd.concat(cv_dfs, ignore_index=True)
print(f"\nCombined pred_df: {pred_df.shape}  ({pred_df['spec_id'].nunique():,} unique spec_ids)")

with open(DATA_ROOT / "proc/nist23/spec_df.pkl", "rb") as f:
    spec_df = pickle.load(f)

with open(DATA_ROOT / "proc/nist23/mol_df.pkl", "rb") as f:
    mol_df = pickle.load(f)
mol_df = mol_df.drop(columns=["mol"], errors="ignore")

with open(DATA_ROOT / "classyfire/nist23_mol_classyfire.pkl", "rb") as f:
    cf_df = pickle.load(f)

print(f"spec_df:  {spec_df.shape}")
print(f"mol_df:   {mol_df.shape}")
print(f"cf_df:    {cf_df.shape}")

del cv_dfs
gc.collect()


  cv0: 85,660 rows
  cv1: 83,574 rows
  cv2: 85,237 rows
  cv3: 85,873 rows
  cv4: 84,535 rows
  cv5: 84,931 rows
  cv6: 85,025 rows
  cv7: 85,048 rows
  cv8: 84,754 rows

Combined pred_df: (764637, 11)  (527,138 unique spec_ids)
spec_df:  (1899597, 22)
mol_df:   (52787, 13)
cf_df:    (52787, 10)


0

In [3]:
pred_df.head(3)

,spec_id,mol_id,pred_mzs,pred_ints,pred_oos_prob,frag_formula_peak_idxs,frag_formula_peak_mzs,index_rebased,cos_sim_0.01,cos_hun,cv_fold
0,418,34523,"[220.0635986328125, 186.10256958007812, 222.07...","[1.2268697023391724, 4.8837103843688965, 999.9...",0.020965,"[30, 36, 38, 58, 407, 411]","[220.0635986328125, 186.10256958007812, 222.07...",True,0.999276,0.999276,0
1,419,34523,"[194.06053161621094, 173.09475708007812, 220.0...","[1.18855881690979, 4.235246658325195, 2.647926...",0.027789,"[1, 4, 30, 32, 36, 38, 40, 58, 147, 315, 407, ...","[194.06053161621094, 173.09475708007812, 220.0...",True,0.986549,0.986549,0
2,420,34523,"[194.06053161621094, 173.09475708007812, 174.1...","[1.0735852718353271, 13.810319900512695, 2.423...",0.031115,"[1, 4, 8, 28, 30, 32, 36, 38, 40, 58, 147, 315...","[194.06053161621094, 173.09475708007812, 174.1...",True,0.877749,0.877749,0


In [4]:
spec_df.head(3)

,spec_id,mol_id,prec_type,inst_type,frag_mode,spec_type,ion_mode,dset,dset_spec_id,col_gas,...,nce,prec_mz,peaks,ri,group_id,ace_extra_1,nce_extra_1,ace_extra_2,nce_extra_2,ce_type
0,0,8061,[M+H]+,FT,HCD,MS2,P,nist23_hr1,3461655,N2,...,44.0,132.0768,"[(53.0135, 2.4), (55.018, 55.2), (55.0292, 4.2...",NaN,0,NaN,NaN,NaN,NaN,single
1,1,8061,[M+H]+,FT,HCD,MS2,P,nist23_hr1,3461656,N2,...,54.0,132.0768,"[(53.0135, 5.6), (55.018, 82.6), (55.0293, 7.1...",NaN,0,NaN,NaN,NaN,NaN,single
2,2,8061,[M+H]+,FT,HCD,MS2,P,nist23_hr1,3461657,N2,...,65.0,132.0768,"[(51.0231, 8.7), (53.0136, 12.1), (55.018, 107...",NaN,0,NaN,NaN,NaN,NaN,single


In [5]:
mol_df.head(3)

,smiles,mol_id,inchikey_s,scaffold,formula,inchi,mw,exact_mw,num_atoms,num_bonds,charge,single_mol,num_radicals
0,*C1(C)C(OC2OC(CO)C(O)C(OC3OC(CO)C(O)C(O)C3O)C2...,0,,O=C(OC1CCCCO1)C12CCCCC1C1=CCC3C4CCC(OC5CC(OC6C...,C47H75*O19,,944.098,943.490255,67,74,0,True,0
1,BrC(Br)(Br)c1cnc2ccccc2n1,1,VCCOPRMZNNMFLI,c1ccc2nccnc2c1,C9H5Br3N2,"InChI=1S/C9H5Br3N2/c10-9(11,12)8-5-13-6-3-1-2-...",380.865,377.800284,14,15,0,True,0
2,BrC1=CCCN(Cc2ccccc2)C1,2,OFHMQSKRHZDSQE,C1=CCN(Cc2ccccc2)CC1,C12H14BrN,InChI=1S/C12H14BrN/c13-12-7-4-8-14(10-12)9-11-...,252.155,251.030962,14,15,0,True,0


## 2. Merge DataFrames

In [6]:
# Merge predictions with spectrum metadata
spec_meta_cols = [
    "spec_id", "mol_id", "prec_type", "inst_type", "frag_mode",
    "ion_mode", "dset", "ace", "nce", "ce_type",
]
df = pred_df.merge(spec_df[spec_meta_cols], on=["spec_id", "mol_id"], how="left")

# Merge with molecule metadata
mol_meta_cols = ["mol_id", "smiles", "formula", "mw", "exact_mw", "num_atoms", "num_bonds", "inchikey_s"]
df = df.merge(mol_df[mol_meta_cols], on="mol_id", how="left")

# Merge with ClassyFire labels
cf_meta_cols = ["mol_id", "superklass", "klass", "subklass", "direct_parent"]
df = df.merge(cf_df[cf_meta_cols], on="mol_id", how="left")

# Remove MoNA spectra (adduct-matching artifact skews wrecall to ~0)
df = df[df["dset"] != "mona23"].copy()

print(f"Merged df (NIST23 only): {df.shape}")
df.head(3)
del pred_df, spec_df, mol_df, cf_df
gc.collect()


Merged df (NIST23 only): (698676, 30)


0

## 1. Top / Bottom Predictions

In [20]:
display_cols = ["spec_id", "mol_id", "smiles", "formula", "prec_type", "inst_type", "cos_sim_0.01", "cos_hun"]

print("=== Top 10 predictions ===")
display(df.nlargest(10, "cos_sim_0.01")[display_cols].reset_index(drop=True))

print("=== Bottom 10 predictions ===")
display(df.nsmallest(10, "cos_sim_0.01")[display_cols].reset_index(drop=True))

=== Top 10 predictions ===


,spec_id,mol_id,smiles,formula,prec_type,inst_type,cos_sim_0.01,cos_hun
0,647170,46162,O=C(O)c1cccc(I)c1,C7H5IO2,[M-H]-,FT,1.000000,1.000000
1,647170,46162,O=C(O)c1cccc(I)c1,C7H5IO2,[M-H]-,FT,1.000000,1.000000
2,747296,18089,CCN(Cc1c(F)cccc1Cl)c1c([N+](=O)[O-])cc(C(F)(F)...,C16H12ClF4N3O4,[M+H]+,FT,1.000000,1.000000
3,1155016,45574,O=C(O)c1cc(Cl)ccc1C(F)(F)F,C8H4ClF3O2,[M-H]-,FT,1.000000,1.000000
4,1252731,22684,COC(=O)C(C)CS(=O)(=O)O,C5H10O5S,[M+Na]+,FT,1.000000,1.000000
5,1188841,51251,OCc1ncccc1F,C6H6FNO,[M+H]+,FT,0.999999,1.000000
6,722931,34132,Cc1ncc(-c2ccnc(Nc3ccc(S(C)(=O)=O)cc3)n2)n1C(C)C,C18H21N5O2S,[M+H]+,FT,0.999999,0.999999
7,1263111,18409,CCNCc1ccccc1,C9H13N,[M+H]+,FT,0.999999,0.999999
8,940532,39589,NNC(=O)CC(F)(F)F,C3H5F3N2O,[M+H]+,FT,0.999999,0.999999
9,551714,27314,COc1ccc(-c2nc3cc([N+](=O)[O-])ccc3[nH]2)cc1,C14H11N3O3,[M-H]-,FT,0.999999,0.999999


=== Bottom 10 predictions ===


,spec_id,mol_id,smiles,formula,prec_type,inst_type,cos_sim_0.01,cos_hun
0,9358,19838,CCOc1cc(=O)oc2ccc(OC)cc12,C12H12O4,[M+H]+,FT,0.0,0.764567
1,9564,13772,CCCC(O)CC,C6H14O,[M+H]+,QTOF,0.0,0.000000
2,9671,23170,COC(=O)CC(C)=O,C5H8O3,[M+H]+,QTOF,0.0,0.057358
3,9672,23170,COC(=O)CC(C)=O,C5H8O3,[M+H]+,QTOF,0.0,0.028581
4,9673,23170,COC(=O)CC(C)=O,C5H8O3,[M+H]+,QTOF,0.0,0.012589
5,9674,23170,COC(=O)CC(C)=O,C5H8O3,[M+H]+,QTOF,0.0,0.006699
6,9675,23170,COC(=O)CC(C)=O,C5H8O3,[M+H]+,QTOF,0.0,0.002818
7,9676,23170,COC(=O)CC(C)=O,C5H8O3,[M+H]+,QTOF,0.0,0.004042
8,9677,23170,COC(=O)CC(C)=O,C5H8O3,[M+H]+,QTOF,0.0,0.000000
9,9678,23170,COC(=O)CC(C)=O,C5H8O3,[M+H]+,QTOF,0.0,0.001769
